In [3]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_C.xpt"
csv_file = DATA_DIR / "HUQ_C_corrected.csv"


# ============================================================
# 2. READ XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

print("=" * 70)
print("HUQ_C XPT -> CSV")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 3. KNOWN ZERO ARTIFACT
# ============================================================

TINY_VALUE = 5.397605346934028e-79

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


# ============================================================
# 4. CHECK BEFORE CORRECTION
# ============================================================

print("\n--- BEFORE CORRECTION ---")

true_zero_count = int(
    df[numeric_cols].eq(0).sum().sum()
)

tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

total_tiny = int(tiny_counts.sum())

print("True numeric zeros:", true_zero_count)
print("Tiny values:", total_tiny)


if total_tiny > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[tiny_counts > 0]
        .sort_values(ascending=False)
    )


# ============================================================
# 5. HUQ050 BEFORE
# ============================================================

if "HUQ050" in df.columns:

    print("\n--- HUQ050 BEFORE ---")

    print(
        "True zeros:",
        int(df["HUQ050"].eq(0).sum())
    )

    print(
        "Tiny values:",
        int(df["HUQ050"].eq(TINY_VALUE).sum())
    )

    print("\nDistribution:")

    print(
        df["HUQ050"]
        .value_counts(dropna=False)
        .sort_index()
    )


# ============================================================
# 6. RESTORE TRUE ZERO
# ============================================================

for col in numeric_cols:

    mask = df[col].eq(TINY_VALUE)

    if mask.any():
        df.loc[mask, col] = 0


# ============================================================
# 7. WRITE WHOLE-NUMBER HUQ VALUES AS INTEGERS
# ============================================================

#
# Examples:
#
# 0.0     -> 0
# 1.0     -> 1
# 99.0    -> 99
# SEQN .0 -> removed
#
# A column is changed only if every nonmissing value
# is a whole number.
#

for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    if np.all(np.isclose(values, np.round(values))):

        df[col] = (
            df[col]
            .round()
            .astype("Int64")
        )


# ============================================================
# 8. HUQ050 AFTER
# ============================================================

if "HUQ050" in df.columns:

    print("\n--- HUQ050 AFTER ---")

    print(
        "dtype:",
        df["HUQ050"].dtype
    )

    print(
        "True zeros:",
        int(df["HUQ050"].eq(0).sum())
    )

    print(
        "Tiny values:",
        int(df["HUQ050"].eq(TINY_VALUE).sum())
    )

    print("\nFinal distribution:")

    print(
        df["HUQ050"]
        .value_counts(dropna=False)
        .sort_index()
    )


# ============================================================
# 9. CHECK SEQN
# ============================================================

if "SEQN" in df.columns:

    print("\n--- SEQN CHECK ---")

    print("dtype:", df["SEQN"].dtype)

    print(
        df["SEQN"]
        .head(10)
        .to_string(index=False)
    )


# ============================================================
# 10. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


# ============================================================
# 11. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

remaining_tiny = int(
    df.select_dtypes(include=[np.number])
    .eq(TINY_VALUE)
    .sum()
    .sum()
)

print(
    "Remaining tiny values:",
    remaining_tiny
)


if "HUQ050" in df.columns:

    huq_zero_count = int(
        df["HUQ050"].eq(0).sum()
    )

    print(
        "HUQ050 zero count:",
        huq_zero_count
    )

    if huq_zero_count == 1250:
        print(
            "PASS: HUQ050 has 1,250 records coded 0."
        )
    else:
        print(
            "CHECK: HUQ050 zero count is not 1,250."
        )


print("\nCSV created:")
print(csv_file)

HUQ_C XPT -> CSV
Rows: 10122
Columns: 16

Columns:
['SEQN', 'HUQ010', 'HUQ020', 'HUQ030', 'HUQ040', 'HUQ050', 'HUQ060', 'HUQ071', 'HUD080', 'HUQ082', 'HUQ084', 'HUQ086', 'HUQ088A', 'HUQ088B', 'HUQ088C', 'HUQ090']

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny values: 1251

Tiny values by variable:
HUQ050    1250
HUQ084       1
dtype: int64

--- HUQ050 BEFORE ---
True zeros: 0
Tiny values: 1250

Distribution:
HUQ050
5.397605e-79    1250
1.000000e+00    2082
2.000000e+00    2987
3.000000e+00    2551
4.000000e+00     615
5.000000e+00     630
9.900000e+01       7
Name: count, dtype: int64

--- HUQ050 AFTER ---
dtype: Int64
True zeros: 1250
Tiny values: 0

Final distribution:
HUQ050
0     1250
1     2082
2     2987
3     2551
4      615
5      630
99       7
Name: count, dtype: Int64

--- SEQN CHECK ---
dtype: Int64
21005
21006
21007
21008
21009
21010
21011
21012
21013
21014

FINAL VALIDATION
Remaining tiny values: 0
HUQ050 zero count: 1250
PASS: HUQ050 has 1,250 records coded 0.

CS